# TCGA-PRAD pilot pipeline

Download → clean → match gene expression to clinical outcomes → QC → save.

**Decisions applied (CampLab client meetings):** progression-free interval (PFI) endpoint; match on the first 12 characters of the TCGA barcode; primary tumors (`01`), one sample per patient; covariates age, pathologic T/N stage and Gleason; genes with missing values removed; QC outliers (|z| > 3) excluded.

The logic lives in `src/tcga_prad_pilot.py`; this notebook runs it one step at a time so each step can be inspected. To run everything in one go instead: `python src/tcga_prad_pilot.py`.

**Colab:** first run `!git clone <repo-url>` and `%cd <repo-name>/notebooks`, then *Runtime → Run all*.

## Setup

Set `MODULE_FILE` to the module gene list (CSV/TSV with a gene column and a module column) to report module coverage.

In [ ]:
import os
import sys

# make src/ importable whether this runs from the repo root or from notebooks/
for path in ("src", "../src"):
    if os.path.isdir(path):
        sys.path.insert(0, os.path.abspath(path))
import tcga_prad_pilot as p

OUT = "pilot_out"
MODULE_FILE = None
Z_THRESHOLD = 3.0
log = p.QCLog()

## 1. Download

Expression: cBioPortal datahub copy of `data_mrna_seq_v2_rsem.txt` (~77 MB). Clinical: cBioPortal API. Gleason: TCGA Firehose Legacy study. Sources and a SHA-256 fingerprint go into `provenance.json`.

In [ ]:
inputs = p.load_inputs()
inputs["provenance"]

## 2. Clean expression

Drop rows without a gene symbol, keep one row per symbol (the most highly expressed), transform to log2(RSEM + 1), and remove genes with missing values (PanCanAtlas batch-correction gaps) so every sample is kept.

In [ ]:
expr, features, removed_genes = p.clean_expression(inputs["expr_raw"], log)
expr.shape

## 3. Select samples (barcode rule + QC)

Characters 1–12 of the barcode = patient; characters 14–15 = sample type. Keep `01` (primary tumor), one sample per patient, and exclude samples more than `Z_THRESHOLD` standard deviations from the mean on median expression or genes detected.

In [ ]:
manifest = p.select_samples(expr, log, Z_THRESHOLD)
manifest[~manifest["included"]]

## 4. Clean clinical data

Endpoint: PFI (`PFS_STATUS` / `PFS_MONTHS`). Secondary: OS, DSS, DFI. Covariates: age, pathologic T and N stage, Gleason.

In [ ]:
clin = p.clean_clinical(inputs["patients"], inputs["samples"], inputs["gleason"], log)
clin.head()

## 5. Match expression to clinical outcomes

Join on the 12-character patient ID. The function stops with an error if a patient appears twice or the two tables are out of order.

In [ ]:
clinical, expression, manifest = p.match(manifest, clin, expr, log)
features = p.annotate_features(features, expression)
clinical.head()

## 6. Module coverage (optional)

In [ ]:
coverage = p.module_coverage(MODULE_FILE, expression.index, removed_genes, log) if MODULE_FILE else None
coverage

## 7. Save deliverables

Three linked tables (expression, clinical, feature annotation), sample manifest, data dictionary, removed-gene list, provenance, QC summary, and an AnnData file.

In [ ]:
p.save_outputs(OUT, expression, clinical, features, manifest, inputs["provenance"], removed_genes, log, coverage)
sorted(os.listdir(OUT))

## Review

In [ ]:
print(log.text())

In [ ]:
clinical.describe(include="all").T